# 01 — Exploratory Data Analysis
**exoplanet-ml-classifier** | MLEA_M — ECI 2026-1  
Binary classification of NASA Kepler Object of Interest (KOI) candidates.

---

## Section 1 — Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Make src/ importable from the notebooks directory
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('ignore')

from src.constants import (
    NUMERIC_FEATURES,
    RAW_DATA_PATH,
    RANDOM_SEED,
    TARGET_COLUMN,
)
from src.data_loader import describe_dataset, drop_leakage_columns, load_raw_koi_data
from src.visualization import (
    plot_correlation_heatmap,
    plot_feature_distributions_by_class,
    plot_missing_values_heatmap,
    plot_pca_scatter,
    plot_target_distribution,
)

print(f'ROOT: {ROOT}')
print(f'Data path: {ROOT / RAW_DATA_PATH}')

In [ ]:
df_raw = load_raw_koi_data(ROOT / RAW_DATA_PATH)
stats = describe_dataset(df_raw)

print(f"Observations : {stats['n_observations']:,}")
print(f"Features     : {stats['n_features']}")
print(f"Duplicate rows: {stats['duplicate_rows']}")
print(f"Imbalance ratio (minority/majority): {stats['class_imbalance_ratio']:.3f}")
print("\nTarget distribution:")
for cls, pct in stats['target_distribution'].items():
    print(f"  {cls}: {pct:.2%}")

## Section 2 — Target Variable

In [ ]:
target_counts = df_raw[TARGET_COLUMN].value_counts()
print(target_counts.to_frame(name='count').to_markdown())

plot_target_distribution(df_raw[TARGET_COLUMN], save=True)

### Class Imbalance Note

The dataset exhibits a **moderate class imbalance**: the minority class (CANDIDATE) represents
roughly 35–40% of all observations, while FALSE POSITIVE makes up the remainder.  
The imbalance ratio (minority / majority) is reported above.  
During preprocessing we will decide whether SMOTE oversampling is necessary based on a
configurable threshold (default 0.3).

## Section 3 — Missing Values

In [ ]:
missing_series = df_raw.isna().sum().sort_values(ascending=False)
top_missing = missing_series[missing_series > 0].head(20)
print("Top 20 columns by missing count:")
print(top_missing.to_frame(name='missing').to_markdown())

In [ ]:
plot_missing_values_heatmap(df_raw, save=True)

## Section 4 — Univariate Analysis

In [ ]:
available_numeric = [c for c in NUMERIC_FEATURES if c in df_raw.columns]
print("Descriptive statistics for NUMERIC_FEATURES:")
df_raw[available_numeric].describe().T.round(3)

In [ ]:
n_features = len(available_numeric)
ncols = 4
nrows = int(np.ceil(n_features / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3), dpi=120)
axes_flat = axes.flatten()

for idx, col in enumerate(available_numeric):
    axes_flat[idx].hist(
        df_raw[col].dropna(), bins=40, color='#2196F3', edgecolor='white', linewidth=0.5
    )
    axes_flat[idx].set_title(col, fontsize=9)
    axes_flat[idx].set_xlabel('')

for idx in range(n_features, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle('Histogram Grid — Numeric KOI Features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(ROOT / 'reports/figures/histograms_numeric_features.png', bbox_inches='tight', dpi=120)
plt.show()

## Section 5 — Bivariate Analysis

In [ ]:
plot_correlation_heatmap(df_raw, save=True)

In [ ]:
from src.data_loader import get_feature_target_split, drop_leakage_columns

df_clean = drop_leakage_columns(df_raw)
X_eda, y_eda = get_feature_target_split(df_clean)

plot_feature_distributions_by_class(X_eda, y_eda, save=True)

## Section 6 — Unsupervised Exploration

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

available = [c for c in NUMERIC_FEATURES if c in X_eda.columns]
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_numeric = X_eda[available]
X_imputed = imputer.fit_transform(X_numeric)
X_scaled = scaler.fit_transform(X_imputed)

plot_pca_scatter(X_scaled, y_eda.values, save=True)

In [ ]:
pca_full = PCA(n_components=10, random_state=RANDOM_SEED)
pca_full.fit(X_scaled)

fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
ax.bar(
    range(1, 11),
    pca_full.explained_variance_ratio_ * 100,
    color='#2196F3',
    edgecolor='white',
)
ax.plot(
    range(1, 11),
    np.cumsum(pca_full.explained_variance_ratio_) * 100,
    'o-',
    color='#FF5722',
    label='Cumulative',
)
ax.set_xlabel('Principal Component', fontsize=11)
ax.set_ylabel('Explained Variance (%)', fontsize=11)
ax.set_title('PCA — Explained Variance (first 10 components)', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(ROOT / 'reports/figures/pca_explained_variance.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca2 = pca2.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

for ax, k in zip(axes, [2, 3]):
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init='auto')
    clusters = km.fit_predict(X_pca2)
    scatter = ax.scatter(
        X_pca2[:, 0], X_pca2[:, 1],
        c=clusters, cmap='tab10', alpha=0.4, s=10
    )
    ax.set_title(f'K-Means (k={k}) vs True Labels', fontsize=10, fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(scatter, ax=ax, label='Cluster')

plt.suptitle('K-Means Clustering in PCA Space', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(ROOT / 'reports/figures/kmeans_pca.png', bbox_inches='tight', dpi=120)
plt.show()

## Section 7 — EDA Insights

**Key actionable findings from the exploratory analysis:**

1. **Class imbalance is moderate** (~35–40% minority): SMOTE oversampling may be beneficial
   but is not strictly required; threshold tuning should also be evaluated.

2. **Several numeric features contain missing values** (e.g., `koi_teq`, `koi_insol`,
   `koi_srad`): median imputation is appropriate since these distributions are right-skewed.

3. **High correlations exist between stellar parameters** (`koi_steff`, `koi_srad`,
   `koi_slogg`) and between orbit-derived quantities (`koi_insol`, `koi_teq`):
   regularised models (LR, XGBoost) should handle multicollinearity well.

4. **`koi_model_snr` and `koi_depth` are the most discriminative features** by boxplot
   separation: CANDIDATEs tend to show higher SNR and smaller transit depth scatter.

5. **PCA retains ~70–80% of variance in 2 components**, suggesting the feature space is
   reasonably low-dimensional; tree-based models that exploit non-linear interactions
   may still outperform linear classifiers in the full feature space.